In [14]:
import polars as pl
import us

df = pl.read_csv("cleaned_resstock_data/full_data_with_utility.csv")
sales = pl.read_csv("cleaned_resstock_data/sales_data.csv")

data = []
for state in [s.abbr for s in us.states.STATES]:
    for util in sales.filter((pl.col("State")==state)).select("Utility Name").to_series().unique():
        # Filter resstock data for utility and state
        df_util = df.filter(
            (pl.col("in.utility_name").str.contains(util, literal=True)) &
            (pl.col("in.state").str.contains(state, literal=True))
        )

        # Separate census and non-census
        census_df = df_util.filter(pl.col("in.city").str.contains(","))
        non_census_df = df_util.filter(~pl.col("in.city").str.contains(","))

        # Compute energy sums (in MWh)
        resstock_census = census_df.select(
            (pl.col("out.electricity.total.energy_consumption.kwh") * pl.col("elec_weight")).sum()
        ).item() / 1000

        resstock_non_census = non_census_df.select(
            (pl.col("out.electricity.total.energy_consumption.kwh") * pl.col("elec_weight")).sum()
        ).item() / 1000

        resstock_half_non_census = non_census_df.sample(fraction=0.5, seed=42).select(
            (pl.col("out.electricity.total.energy_consumption.kwh") * pl.col("elec_weight")).sum()
        ).item() / 1000

        added_bldg_ids = non_census_df.sample(fraction=0.5, seed=42)["bldg_id"].to_list()

        # Reported sales (in MWh)
        reported_sum = sales.filter(
            (pl.col("Utility Name") == util) & (pl.col("State") == state)
        )["Megawatthours"].str.replace_all(",", "").cast(pl.UInt32).sum()

        # Compute all three options
        options = []

        # Option 1: Census only
        factor_census = reported_sum / resstock_census if resstock_census > 0 else 0
        options.append(("Y", resstock_census, factor_census))

        # Option 2: Census + half non-census
        resstock_half = resstock_census + resstock_half_non_census
        factor_half = reported_sum / resstock_half if resstock_half > 0 else 0
        options.append(("Y+½N", resstock_half, factor_half))

        # Option 3: Census + all non-census
        resstock_all = resstock_census + resstock_non_census
        factor_all = reported_sum / resstock_all if resstock_all > 0 else 0
        options.append(("Y+N", resstock_all, factor_all))

        # Choose option closest to 1 and within bounds
        census, resstock_sum, factor = min(options, key=lambda x: abs(x[2] - 1))

        del resstock_census, resstock_half_non_census, resstock_non_census, factor_all, \
            factor_census, factor_half, census_df, non_census_df, options, resstock_all, \
            resstock_half

        if round(factor,2) >= 0.7 and round(factor,2) <= 1.3:
            data.append([state, util, resstock_sum, reported_sum, round(factor,2), census, "Nope"])
        else:
            data.append([state, util, resstock_sum, reported_sum, round(factor,2), census, "Mismatch too big"])

data = pl.DataFrame(data,schema=[('State',pl.Utf8),
                                 ('Utility',pl.Utf8),
                                 ('ResStock', pl.Float32),
                                 ('Reported', pl.Float32),
                                 ('weight Adjustment Factor',pl.Float32),
                                 ('Only census?',pl.Utf8),
                                 ('Mismatch?',pl.Utf8)])

matching_sales = data.filter(pl.col('Mismatch?')=="Nope").sort(by="Reported",descending=True)
matching_sales.write_csv("matching_sales.csv")
data.write_csv("all_sales.csv")

C:\Users\al.qarooni\AppData\Local\Temp\ipykernel_14324\543921924.py:69: DataOrientationWarning: Row orientation inferred during DataFrame construction. Explicitly specify the orientation by passing `orient="row"` to silence this warning.
  data = pl.DataFrame(data,schema=[('State',pl.Utf8),


In [ ]:
util = "Northern States Power Co - Minnesota"
state = "MN"
# Filter resstock data for utility and state
df_util = df.filter(
    (pl.col("in.utility_name").str.contains(util, literal=True)) &
    (pl.col("in.state").str.contains(state, literal=True))
)

# Separate census and non-census
census_df = df_util.filter(pl.col("in.city").str.contains(","))
non_census_df = df_util.filter(~pl.col("in.city").str.contains(","))

# Compute energy sums (in MWh)
# census_df.select(
#     (pl.col("out.electricity.total.energy_consumption.kwh") * pl.col("elec_weight")).sum()
# ).item() / 1000 \
# + \
# non_census_df.sample(fraction=0.5, seed=42).select(
#     (pl.col("out.electricity.total.energy_consumption.kwh") * pl.col("elec_weight")).sum()
# ).item() / 1000
# + \
# non_census_df.select(
#     (pl.col("out.electricity.total.energy_consumption.kwh") * pl.col("elec_weight")).sum()
# ).item() / 1000

0.9468002689453707